## Preprocessing and feature engineering

In [ ]:
from pathlib import Path
import sys

In [ ]:
# Resolve the repository root so local src modules can be imported consistently.
PROJECT_ROOT = None

for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / "src").is_dir() and (folder / "data").is_dir():
        PROJECT_ROOT = folder
        break

if PROJECT_ROOT is None:
    raise RuntimeError("Could not find the project root folder. Make sure you open this notebook inside the project folder..")

sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
from src.data_loader import load_model_dataset
from src.preprocessing import chronological_split 
from src.features import FEATURE_COLUMNS, build_supervised_features

# Load the validated arrival/indicator join that supplies both targets and predictors.
monthly = load_model_dataset()

In [ ]:
# Build one row per eligible target month using only values available beforehand.
supervised = build_supervised_features(monthly)
print(f"Supervised target rows: {len(supervised)}")
print(f"Prediction columns: {FEATURE_COLUMNS}")

In [ ]:
# Inspect target dates, predictor columns, and the response together.
display(
    supervised[["Target_Date", *FEATURE_COLUMNS, "Target_Arrivals"]].head()
)

In [ ]:
# Keep the final 18 target months as a future-like holdout; do not shuffle time series.
train, test = chronological_split(supervised, test_months=18)
print(f"Training rows: {len(train)}")
print(f"Training targets: {train["Target_Date"].min():%y-%m} to {train["Target_Date"].max():%y-%m}")
print(f"Testing datas: {len(test)}")
print(f"Testing datas targets: {train["Target_Date"].min():%y-%m} to {train["Target_Date"].max():%y-%m}")